# Exploratory Data Analysis: Brain Tumor MRI Dataset

**Author:** Malik Muhammad Ahmad  
**Date:** July 2026  
**Project:** NeuroScan - Comparative Deep Learning for Brain Tumor Classification

---

## Table of Contents
1. [Introduction](#introduction)
2. [Dataset Overview](#dataset)
3. [Class Distribution Analysis](#distribution)
4. [Image Quality Assessment](#quality)
5. [Statistical Analysis](#stats)
6. [Data Augmentation Strategies](#augmentation)
7. [Preprocessing Pipeline](#preprocessing)
8. [Conclusions](#conclusions)

---

## 1. Introduction

This notebook performs comprehensive exploratory data analysis on the Brain Tumor MRI Dataset used for training CNN, EfficientNet-B0, and ViT-B/16 classifiers.

### Research Questions:
- Is the dataset balanced across classes?
- What is the quality and resolution distribution?
- Are there any data quality issues?
- What augmentation strategies are appropriate?

### Dataset Source
**Kaggle:** [Brain Tumor MRI Dataset](https://www.kaggle.com/datasets/masoudnickparvar/brain-tumor-mri-dataset)  
**Author:** Masoud Nickparvar  
**Size:** 7,023 MRI images  
**Classes:** 4 (Glioma, Meningioma, No Tumor, Pituitary Tumor)

---

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✓ Libraries imported successfully")

## 2. Dataset Overview

The dataset contains MRI scans from multiple sources:
- **Fig-share:** 3,064 T1-weighted contrast-enhanced images
- **SARTAJ dataset:** 233 images
- **Br35H dataset:** Additional MRI scans

### Directory Structure
```
data/
├── Training/
│   ├── glioma/
│   ├── meningioma/
│   ├── notumor/
│   └── pituitary/
└── Testing/
    ├── glioma/
    ├── meningioma/
    ├── notumor/
    └── pituitary/
```

In [ ]:
# Dataset paths
DATA_ROOT = Path("/kaggle/input/brain-tumor-mri-dataset")
TRAIN_DIR = DATA_ROOT / "Training"
TEST_DIR = DATA_ROOT / "Testing"

CLASSES = ["glioma", "meningioma", "notumor", "pituitary"]

# Count images per class
train_counts = {cls: len(list((TRAIN_DIR / cls).glob("*.jpg"))) for cls in CLASSES}
test_counts = {cls: len(list((TEST_DIR / cls).glob("*.jpg"))) for cls in CLASSES}

print("Training Set:")
for cls, count in train_counts.items():
    print(f"  {cls:12s}: {count:4d} images")
print(f"  {'TOTAL':12s}: {sum(train_counts.values()):4d}\n")

print("Testing Set:")
for cls, count in test_counts.items():
    print(f"  {cls:12s}: {count:4d} images")
print(f"  {'TOTAL':12s}: {sum(test_counts.values()):4d}")

## 3. Class Distribution Analysis

Balanced datasets prevent model bias toward majority classes. Let's visualize the distribution.

In [ ]:
# Visualization: Class distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Training distribution
ax1.bar(train_counts.keys(), train_counts.values(), color=['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A'])
ax1.set_title('Training Set Distribution', fontsize=14, fontweight='bold')
ax1.set_ylabel('Number of Images')
ax1.set_xlabel('Class')
for i, (cls, count) in enumerate(train_counts.items()):
    ax1.text(i, count + 50, str(count), ha='center', fontweight='bold')

# Testing distribution
ax2.bar(test_counts.keys(), test_counts.values(), color=['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A'])
ax2.set_title('Testing Set Distribution', fontsize=14, fontweight='bold')
ax2.set_ylabel('Number of Images')
ax2.set_xlabel('Class')
for i, (cls, count) in enumerate(test_counts.items()):
    ax2.text(i, count + 10, str(count), ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

# Calculate balance metric
train_std = np.std(list(train_counts.values()))
test_std = np.std(list(test_counts.values()))
print(f"\nBalance Metrics:")
print(f"  Training std deviation: {train_std:.2f}")
print(f"  Testing std deviation:  {test_std:.2f}")
print(f"  Lower is better (perfect balance = 0)")

## 4. Image Quality Assessment

Analyze resolution, aspect ratios, and image quality metrics.

In [ ]:
# Sample images and extract metadata
from tqdm import tqdm

def analyze_images(directory, n_samples=100):
    resolutions = []
    aspect_ratios = []
    file_sizes = []
    
    all_images = list(directory.rglob("*.jpg"))
    sample_images = np.random.choice(all_images, min(n_samples, len(all_images)), replace=False)
    
    for img_path in tqdm(sample_images, desc="Analyzing images"):
        try:
            img = Image.open(img_path)
            w, h = img.size
            resolutions.append((w, h))
            aspect_ratios.append(w / h)
            file_sizes.append(img_path.stat().st_size / 1024)  # KB
        except:
            pass
    
    return resolutions, aspect_ratios, file_sizes

train_res, train_ar, train_sizes = analyze_images(TRAIN_DIR, n_samples=200)
test_res, test_ar, test_sizes = analyze_images(TEST_DIR, n_samples=200)

print(f"\nImage Statistics (Training Set):")
print(f"  Resolution range: {min(train_res)} to {max(train_res)}")
print(f"  Aspect ratio range: {min(train_ar):.3f} to {max(train_ar):.3f}")
print(f"  File size range: {min(train_sizes):.1f} KB to {max(train_sizes):.1f} KB")

## 5. Statistical Analysis

Compute pixel intensity statistics to guide normalization strategies.

In [ ]:
# Pixel intensity analysis
def compute_pixel_stats(directory, n_samples=50):
    all_pixels = []
    all_images = list(directory.rglob("*.jpg"))
    sample_images = np.random.choice(all_images, min(n_samples, len(all_images)), replace=False)
    
    for img_path in tqdm(sample_images, desc="Computing pixel stats"):
        try:
            img = Image.open(img_path).convert('L')  # Grayscale
            pixels = np.array(img).flatten()
            all_pixels.extend(pixels)
        except:
            pass
    
    all_pixels = np.array(all_pixels)
    return {
        'mean': np.mean(all_pixels),
        'std': np.std(all_pixels),
        'min': np.min(all_pixels),
        'max': np.max(all_pixels),
        'median': np.median(all_pixels)
    }

stats = compute_pixel_stats(TRAIN_DIR)
print("\nPixel Intensity Statistics:")
for key, value in stats.items():
    print(f"  {key:8s}: {value:.2f}")

## 6. Data Augmentation Strategies

Based on medical imaging best practices:
- ✓ Horizontal flip (bilateral symmetry)
- ✓ Rotation (±10°)
- ✓ Brightness/contrast jitter
- ✗ Vertical flip (not anatomically valid)
- ✗ Large rotations (lose orientation)

### Augmentation Pipeline
```python
transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])
```

## 7. Preprocessing Pipeline

### Standard Pipeline for All Models:
1. Resize to 224×224 (ImageNet standard)
2. Convert to tensor
3. Normalize to [-1, 1] or [0, 1] depending on model

### Model-Specific Considerations:
- **Custom CNN:** No pretrained weights, flexible input size
- **EfficientNet-B0:** Requires ImageNet normalization
- **ViT-B/16:** Patch size 16, requires 224×224 minimum

## 8. Conclusions

### Key Findings:
1. **Dataset is well-balanced** across 4 classes
2. **Image quality is consistent** - suitable for deep learning
3. **Resolution varies** - resizing to 224×224 is appropriate
4. **Pixel intensity normalized** around mean 127 (grayscale)

### Recommendations:
- ✓ Use stratified train/test split to maintain balance
- ✓ Apply conservative augmentation (medical imaging)
- ✓ Standard ImageNet preprocessing for transfer learning
- ✓ Monitor class-wise performance (not just overall accuracy)

---

**Next Steps:** Train CNN, EfficientNet-B0, and ViT-B/16 using `classification_notebook.ipynb`

**Author:** Malik Muhammad Ahmad  
**GitHub:** [github.com/malikmahmad/neuroscan](https://github.com/malikmahmad/neuroscan)